In [16]:
import pandas as pd
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [11]:
train.head()

,ID,date,sp500 open,sp500 high,sp500 low,sp500 close,sp500 volume,sp500 high-low,nasdaq open,nasdaq high,nasdaq low,nasdaq close,nasdaq volume,nasdaq high-low,us_rates_%,CPI,usd_chf,eur_usd,GDP,silver open,silver high,silver low,silver close,silver volume,silver high-low,oil open,oil high,oil low,oil close,oil volume,oil high-low,platinum open,platinum high,platinum low,platinum close,platinum volume,platinum high-low,palladium open,palladium high,palladium low,palladium close,palladium volume,palladium high-low,gold open,gold high,gold low,gold close,gold volume
0,R00210,2010-11-11,121.05,121.82,120.68,121.64,157659616.0,1.14,52.90,53.480,52.66,53.385,85163600.0,0.820,NaN,NaN,NaN,NaN,NaN,27.40,27.6698,26.8400,27.62,372618.0,0.8298,37.90,38.04,37.76,37.84,8849109.0,0.28,174.45,174.7000,173.2600,174.64,39800.0,1.4400,71.30,71.48,69.4500,71.07,279282.0,2.0300,137.6200,137.75,136.45,137.66,15380502.0
1,R03365,2023-05-26,415.33,420.77,415.25,420.02,93829975.0,5.52,340.76,349.245,340.66,348.400,63006952.0,8.585,NaN,NaN,0.9060,1.0713,NaN,22.21,22.3850,22.0822,22.35,389291.0,0.3028,64.90,64.99,64.36,64.80,2145281.0,0.63,94.79,95.6600,94.4500,94.45,83380.0,1.2100,132.36,133.61,131.3700,131.37,20046.0,2.2400,181.0100,181.30,180.09,180.92,5823674.0
2,R03382,2023-06-22,433.95,436.62,433.60,436.51,70637175.0,3.02,360.63,366.330,360.22,366.170,47603043.0,6.110,NaN,NaN,0.8967,1.0953,NaN,21.56,21.6400,21.3836,21.44,409222.0,0.2564,63.15,63.65,62.14,62.52,3882134.0,1.51,86.63,86.8169,85.3300,85.33,164282.0,1.4869,122.23,122.23,116.9100,118.86,74797.0,5.3200,178.3600,178.99,177.63,177.71,7948565.0
3,R01294,2015-03-06,210.46,210.46,207.10,207.50,188127982.0,3.36,108.50,108.710,107.14,107.410,30990366.0,1.570,NaN,NaN,0.9846,1.0855,NaN,15.65,15.6800,15.5200,15.63,141738.0,0.1600,18.41,18.55,17.97,18.24,27455956.0,0.58,112.97,112.9700,112.2500,112.35,64345.0,0.7200,79.20,79.73,79.0500,79.28,60988.0,0.6800,113.1700,113.26,111.70,111.86,11190846.0
4,R02502,2019-12-20,320.46,321.97,319.39,320.73,149214080.0,2.58,211.86,212.520,211.27,211.710,27690934.0,1.250,NaN,NaN,0.9824,1.1076,NaN,16.65,16.7200,16.5700,16.64,92711.0,0.1500,12.72,12.72,12.56,12.63,11434745.0,0.16,88.25,88.3100,85.6471,85.81,65233.0,2.6629,182.57,182.57,172.1033,173.79,88426.0,10.4667,139.3705,139.52,138.98,139.52,4446611.0


In [18]:
# Fill missing columns with forward fill (previous values)
# This is optimal here as these values stay constant on short periods of time
# Also transform date into proper format
def transformData(df):
    df['date'] = pd.to_datetime(df['date'])
    # Sorting by date helps us later when filling missing values with previous data
    df = df.sort_values('date')

    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.dayofweek

    df = df.drop(columns=['date']) # Get rid of the old format
    # Bfill is just a safety net for missing values at the start of the dataset
    df = df.ffill().bfill()
    # Mantain the original order
    df = df.sort_index()

    return df

x_train = transformData(train)
y_train = x_train['gold close']
x_train = x_train.drop(columns=['ID', 'gold close'])

x_test = transformData(test)
x_test = x_test.drop(columns=['ID'])

pipeline = Pipeline([
    ('model', XGBRegressor(random_state=42))
])

param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [3, 5],
    'model__learning_rate': [0.05, 0.1]
}

gs = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    scoring='neg_root_mean_squared_error',
    cv=5
)
gs.fit(x_train, y_train)
model = gs.best_estimator_
predictions = model.predict(x_test)

In [21]:
rows = []
for id, pred in zip(test['ID'], predictions):
    rows.append({
        'ID': id,
        'gold close': pred
    })
sub = pd.DataFrame(rows)
sub.to_csv('submission.csv', index=False)